In [1]:
import pandas as pd
import os
import numpy as np
# 定义固定列名
fixed_columns = ['hdok', 'plok', 'none', 'nums', 'totalNums', 'average_rssi', 'snr', 'sf', 'tp', 'serial_size']

# 读取 TXT 文件，遍历 data\rawData 文件夹下的所有 txt 文件
file_path = r'data\rawData\FLOOR3'
save_file_path = r'data\processedData\FLOOR3'
new_path = r'd:\Desktop\PHD\reasearch\biyework\maml'
os.chdir(new_path)
# 检查当前路径是否切换成功
current_path = os.getcwd()
print(current_path)

d:\Desktop\PHD\reasearch\biyework\maml


In [4]:
max_serial_size = 0
if not os.path.exists(file_path):
    print(f"文件夹 {file_path} 不存在")
else:
    files = os.listdir(file_path)
    # 遍历 rawData/FLOOR3 文件夹下的所有文件夹，然后再遍历文件夹下的所有 txt 文件
    for folder in files:
        folder_path = os.path.join(file_path, folder)
        if os.path.isdir(folder_path):
            txt_files = os.listdir(folder_path)
            for txt_file in txt_files:
                if txt_file.endswith('.txt'):
                    # 读取 TXT 文件
                    with open(os.path.join(folder_path, txt_file), 'r', encoding='utf-8') as f:
                        lines = f.readlines()
                    
                    # 读取数据
                    data = []
                    for line in lines:
                        if line.startswith('HDOK') or line.startswith('PLOK'):
                            line = line.strip().split()
                            data.append(line)
                    
                    if data:
                        # serial_size可能有误，所以根据实际情况自动剔除data中每一行最后多出来的几个数据
                        serial_size = int(data[0][9])
                        max_serial_size = max(max_serial_size, serial_size)
                        # 确保每一行的数据列数与列名数量一致
                        for i in range(len(data)):
                            if len(data[i]) > len(fixed_columns) + serial_size:
                                data[i] = data[i][:len(fixed_columns) + serial_size]
                        # 动态生成列名
                        columns = fixed_columns + [f'rssi_{i}' for i in range(serial_size)]
                        # 在列尾加上一个新的列名，对应的是txt文件名，且不包含后缀，用于标识数据来源
                        columns.append('location_id')
                        # 将 txt 文件名添加到数据中
                        for i in range(len(data)):
                            data[i].append(txt_file.replace('.txt', '').replace('.0', ''))  #location_id列如果存在.0结尾的数字，则去掉.0
                            
                        # 转换为 DataFrame
                        df = pd.DataFrame(data, columns=columns)
                        # 计算 rssi 序列的方差，并保存在csv文件中
                        rssi_columns = [f'rssi_{i}' for i in range(serial_size)]
                        
                        #计算平均值
                        # Ensure the RSSI columns are converted to numeric before calculating the mean
                        df[rssi_columns] = df[rssi_columns].apply(pd.to_numeric, errors='coerce')
                        df['realtime_average_rssi'] = df[rssi_columns].mean(axis=1)
                        columns.append('realtime_average_rssi')
                        
                        # 计算rssi的中位数
                        df['median_rssi'] = df[rssi_columns].median(axis=1)
                        columns.append('median_rssi')
                        
                        # 计算rssi的众数
                        df['mode_rssi'] = df[rssi_columns].mode(axis=1)[0]
                        columns.append('mode_rssi')
                        
                        # 计算方差前除去异常值 最小的和最大的各舍弃一个进行计算
                        # 去除每一行的最小值和最大值后计算方差
                        def calculate_variance_without_outliers(row):
                            # 将当前行的 RSSI 数据转换为浮点数
                            rssi_values = row[rssi_columns].astype(float).values
                            # 去除最小值和最大值
                            if len(rssi_values) > 2:  # 确保至少有三个值，否则无法去除最小值和最大值
                                rssi_values = np.sort(rssi_values)[4:-4]
                            # 计算方差
                            return np.var(rssi_values)
                        df['rssi_variance'] = df.apply(calculate_variance_without_outliers, axis=1)
                        columns.append('rssi_variance')
                        
                        # 保存为 CSV 文件，并且保存在 'data\processedData' 文件夹下
                        save_folder_path = os.path.join(save_file_path, folder)
                        if not os.path.exists(save_folder_path):
                            os.makedirs(save_folder_path)
                        df.to_csv(os.path.join(save_folder_path, txt_file.replace('.txt', '.csv')), index=False)
                        print(f"文件 {txt_file} 转换成功")
      

文件 1m.txt 转换成功
文件 302.txt 转换成功
文件 306-304.txt 转换成功
文件 308.txt 转换成功
文件 312.txt 转换成功
文件 318.txt 转换成功
文件 322.txt 转换成功
文件 328.txt 转换成功
文件 330.txt 转换成功
文件 334.txt 转换成功
文件 336.txt 转换成功
文件 340.txt 转换成功
文件 344.txt 转换成功
文件 348.txt 转换成功
文件 354.txt 转换成功
文件 356.txt 转换成功
文件 360.txt 转换成功
文件 366.txt 转换成功
文件 370.txt 转换成功
文件 point1.txt 转换成功
文件 point2.txt 转换成功
文件 point3.txt 转换成功
文件 1m.txt 转换成功
文件 302.txt 转换成功
文件 304.txt 转换成功
文件 308.txt 转换成功
文件 312.txt 转换成功
文件 318.txt 转换成功
文件 322.txt 转换成功
文件 328.txt 转换成功
文件 330.txt 转换成功
文件 334.txt 转换成功
文件 336.txt 转换成功
文件 340.txt 转换成功
文件 344.txt 转换成功
文件 348.txt 转换成功
文件 354.txt 转换成功
文件 356.txt 转换成功
文件 360.txt 转换成功
文件 366.txt 转换成功
文件 370.txt 转换成功
文件 point1.txt 转换成功
文件 point2.txt 转换成功
文件 point3.txt 转换成功
文件 1m.txt 转换成功
文件 302.txt 转换成功
文件 306-304.txt 转换成功
文件 308.txt 转换成功
文件 312.txt 转换成功
文件 322.txt 转换成功
文件 328.txt 转换成功
文件 330.txt 转换成功
文件 334.txt 转换成功
文件 336.txt 转换成功
文件 340.txt 转换成功
文件 344.txt 转换成功
文件 350.txt 转换成功
文件 354.txt 转换成功
文件 356.txt 转换成功
文件 360.txt 转换成功
文件 366.txt 转换成功
文

In [5]:
#将转换好的csv文件合并到一个文件中
# 读取 CSV 文件，遍历 data\processedData 文件夹下的所有 csv 文件
save_file_path = r'data\processedData\FLOOR3'
if not os.path.exists(save_file_path):
    print(f"文件夹 {save_file_path} 不存在")
else:
    files = os.listdir(save_file_path)
    # 保存到一个文件中
    all_data = pd.DataFrame()
    for folder in files:
        folder_path = os.path.join(save_file_path, folder)
        if os.path.isdir(folder_path):
            csv_files = os.listdir(folder_path)
            for csv_file in csv_files:
                if csv_file.endswith('.csv'):
                    # 读取 CSV 文件
                    df = pd.read_csv(os.path.join(folder_path, csv_file))
                    all_data = pd.concat([all_data, df], ignore_index=True)
    all_data.to_csv(os.path.join(save_file_path, 'all_data.csv'), index=False)
    print(f"所有文件合并成功")

所有文件合并成功


In [6]:
#把合并好的文件里的rssi_0到rssi_32的数据全都单独提取出来并且和location_id,snr,sf,tp这几个属性单独构成一行
#例如 rssi_0 snr tp sf location_id, rssi_1 snr tp sf location_id, rssi_2 snr tp sf location_id  ... rssi_32 snr tp sf location_id
# 读取 CSV 文件
if not os.path.exists(save_file_path):
    print(f"文件夹 {save_file_path} 不存在")
else:
    # 读取合并后的 CSV 文件
    df = pd.read_csv(os.path.join(save_file_path, 'all_data.csv'))
    # 保存到一个文件中
    new_data=[]
    # 遍历df中的每一行
    for _,row in df.iterrows():
        #如果row['location_id']为浮点数且不为空，则转换为整数
        if isinstance(row['location_id'], str) and not pd.isna(row['location_id']):
            row['location_id'] = row['location_id'].replace('.0', '')
        if isinstance(row['location_id'], float) and not pd.isna(row['location_id']):
            row['location_id'] = int(row['location_id'])
        location_id = row['location_id']
        median_rssi = row['median_rssi']
        realtime_average_rssi = row['realtime_average_rssi']
        average_rssi = row['average_rssi']
        snr = row['snr']
        average_snr=row['snr']
        sf=row['sf']
        tp=row['tp']
        var=row['rssi_variance']
        mode=row['mode_rssi']

        for i in range(max_serial_size):
            rssi=row[f'rssi_{i}']
            #构造新的一行数据
            new_data.append([realtime_average_rssi,average_rssi,var,snr,median_rssi,mode,sf,tp,location_id])
            # new_data.append([rssi,average_rssi,var,snr,sf,tp,location_id])

    new_df=pd.DataFrame(new_data,columns=['realtime_average_rssi','average_rssi','rssi_variance','snr','median_rssi',"mode_rssi",'sf','tp','location_id'])    
    # new_df=pd.DataFrame(new_data,columns=['rssi','average_rssi','rssi_variance','snr','sf','tp','location_id'])

    #将new_df里的空数据行删除
    new_df=new_df.dropna()
    # new_df里面的重复数据去重
    new_df=new_df.drop_duplicates()
    
    #对每一行的数据进行数据增强，生成原来数据的五倍 todo：12是不是有些太大了
    new_data = []
    for _, row in new_df.iterrows():
        # 生成五倍数据
        for i in range(24):
            new_row = row.copy()
            # 对除了location_id以外的每一列数据进行增强
            for col in new_row.index:
                if col != 'location_id' and col != 'sf' and col != 'tp':
                    # 这里可以添加数据增强的逻辑，例如添加噪声、平移等
                    new_row[col] += np.random.normal(0, 1)  # 添加高斯噪声
            new_data.append(new_row)
    # 将增强后的数据转换为 DataFrame
    new_df = pd.DataFrame(new_data, columns=new_df.columns)
    
    #保存到一个新的文件中
    new_df.to_csv(os.path.join(save_file_path, 'all_data_new.csv'), index=False)
    
    print(f"新文件生成成功")

新文件生成成功


不单独把单个rssi列出来,算拟合用的

In [ ]:
#把合并好的文件里的rssi_0到rssi_32的数据全都单独提取出来并且和location_id,snr,sf,tp这几个属性单独构成一行
#例如 rssi_0 snr tp sf location_id, rssi_1 snr tp sf location_id, rssi_2 snr tp sf location_id  ... rssi_32 snr tp sf location_id
# 读取 CSV 文件
if not os.path.exists(save_file_path):
    print(f"文件夹 {save_file_path} 不存在")
else:
    # 读取合并后的 CSV 文件
    df = pd.read_csv(os.path.join(save_file_path, 'all_data.csv'))
    # 保存到一个文件中
    new_data=[]
    # 遍历df中的每一行
    for _,row in df.iterrows():
        if isinstance(row['location_id'], str) and not pd.isna(row['location_id']):
            row['location_id'] = row['location_id'].replace('.0', '')
        if isinstance(row['location_id'], float) and not pd.isna(row['location_id']):
            row['location_id'] = int(row['location_id'])
        average_rssi = row['average_rssi']
        snr = row['snr']
        average_snr=row['snr']
        sf=row['sf']
        tp=row['tp']
        var=row['rssi_variance']
        location_id = row['location_id']
        new_data.append([average_rssi,var,snr,average_snr,sf,tp,location_id])
    new_df=pd.DataFrame(new_data,columns=['average_rssi','rssi_variance','snr','average_snr','sf','tp','location_id'])
    #将new_df里的空数据行删除
    new_df=new_df.dropna()
    #保存到一个新的文件中
    new_df.to_csv(os.path.join(save_file_path, 'all_data_new_fit.csv'), index=False)
    
    print(f"新文件生成成功")

新文件生成成功
